Copyright Hewlett Packard Enterprise Development LP.



In [9]:
import pandas as pd

def duplicate_parquet_with_time_increment(input_path, output_path, multiplier):
    df = pd.read_parquet(input_path)
    if 'time' not in df.columns:
        raise ValueError("The input file must have a 'time' column.")

    # Find the max time in the original data
    df['time'] = pd.to_numeric(df["time"])
    max_time = df['time'].max()
    min_time = df['time'].min()
    time_range = max_time - min_time + 1  # +1 to avoid overlap

    dfs = []
    for i in range(multiplier):
        df_copy = df.copy()
        # Increment time for each duplicate block
        df_copy['time'] = df_copy['time'] + i * time_range
        dfs.append(df_copy)

    df_big = pd.concat(dfs, ignore_index=True)
    df_big.to_parquet(output_path)
    print(f"Duplicated data written to {output_path}")

# Example usage:
# duplicate_parquet_with_time_increment("input.parquet", "output.parquet", 10)

In [ ]:
import arkouda as ak
ak.connect("nodename")

def duplicate_parquet_with_time_increment_ak(input_path, output_path, multiplier):
    df = ak.read_parquet(input_path)
    df = ak.DataFrame(df)
    if 'time' not in df.columns:
        raise ValueError("The input file must have a 'time' column.")

    df["time"] = df["time"].astype(ak.int64) # Convert time to seconds
    max_time = df['time'].max()
    min_time = df['time'].min()
    time_range = max_time - min_time + 1  # +1 to avoid overlap

    df_big = df.copy()
    for i in range(1, multiplier):
        df_copy = df.copy()
        df_copy['time'] = df_copy['time'] + i * time_range
        df_big = df_big.append(df_copy)
        df_copy = None

    df_big.to_parquet(output_path)
    print(f"Duplicated data written to {output_path}")

# Example usage:
# duplicate_parquet_with_time_increment_ak("input.parquet", "output.parquet", 10)

connected to arkouda server tcp://*:5555


In [11]:
# import arkouda as ak
# ak.connect("x1003c6s1b0n0")
# dir = "/lus/scratch/khandeka/dev/arkouda-telemetry-analysis/parquet-traces-for-LSMS-application-16x/"
# power_cap200 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_200_16_events*"
# df = ak.read_parquet(dir + power_cap200)
# df = ak.DataFrame(df)
# df.to_pandas()

In [12]:
import os
import glob

indir = "/lus/scratch/khandeka/dev/arkouda-telemetry-analysis/parquet-traces-for-LSMS-application/"
factor = 2
outdir = f"/lus/scratch/khandeka/dev/arkouda-telemetry-analysis/parquet-traces-for-LSMS-application-{factor}x/"

# List all parquet files in the input directory
power_cap200 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_200_16_events*"
power_cap400 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_400_16_events*"
power_cap300 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_300_16_events*"
power_cap500 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_500_16_events*"
power_cap600 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_600_16_events*"
power_cap700 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_700_16_events*"
power_cap800 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_800_16_events*"
power_cap900 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_900_16_events*"
power_cap1000 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_1000_16_events*"

event_files = [power_cap200, power_cap300, power_cap400, power_cap500, power_cap600, power_cap700, power_cap800, power_cap900, power_cap1000]

# Ensure we have found some files
if not event_files:
    raise FileNotFoundError(f"No files matching the pattern in {indir}. Please check the directory and patterns.")

# Create output directory if it doesn't exist
os.makedirs(outdir, exist_ok=True)
# Check if the output directory is empty
if os.listdir(outdir):
    # Warn and force after a bit of a delay
    print(f"Warning: {outdir} is not empty. Proceeding with caution.")
    import time
    time.sleep(5)
    print("Continuing with the operation...")

print(f"Duplicating {len(event_files)} files from {indir} to {outdir} with factor {factor}...")

for infile in event_files:
    outfile = os.path.join(outdir, os.path.basename(infile))
    print(f"Processing {infile} -> {outfile}")
    duplicate_parquet_with_time_increment_ak(indir + infile, outfile, factor)

print("Duplication complete.")

Duplicating 9 files from /lus/scratch/khandeka/dev/arkouda-telemetry-analysis/parquet-traces-for-LSMS-application/ to /lus/scratch/khandeka/dev/arkouda-telemetry-analysis/parquet-traces-for-LSMS-application-2x/ with factor 2...
Processing paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_200_16_events* -> /lus/scratch/khandeka/dev/arkouda-telemetry-analysis/parquet-traces-for-LSMS-application-2x/paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_200_16_events*


File written successfully!
Duplicated data written to /lus/scratch/khandeka/dev/arkouda-telemetry-analysis/parquet-traces-for-LSMS-application-2x/paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_200_16_events*
Processing paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_300_16_events* -> /lus/scratch/khandeka/dev/arkouda-telemetry-analysis/parquet-traces-for-LSMS-application-2x/paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_300_16_events*
File written successfully!
Duplicated data written to /lus/scratch/khandeka/dev/arkouda-telemetry-analysis/parquet-traces-for-LSMS-application-2x/paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_300_16_events*
Processing paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_400_16_events* -> /lus/scratch/khandeka/dev/arkouda-telemetry-analysis/parquet-traces-for-LSMS-application-2x/paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_400_16_events*
File written successfully!
Duplica